In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
from scipy import stats
from scipy.stats import ttest_rel, ttest_ind
import os

def load_smi_file(filepath):
    """Load SMI data from a directory containing _layer_smi_results.npy file"""
    for file in os.listdir(filepath):
        if file.endswith('_layer_smi_results.npy'):
            layersmi_filepath = os.path.join(filepath, file)
            # load the .npy file
            layersmi_data = np.load(layersmi_filepath, allow_pickle=True).item()
            return layersmi_data                        
    return None

def extract_smi_data_from_results(smi_results, layer_name):
    """
    Extract SMI values for a specific layer from the loaded results
    
    Parameters:
    smi_results: The result from load_smi_file()
    layer_name: String like 'L2/3', 'L4', 'L5', 'L6'
    
    Returns:
    numpy array of SMI values for that layer
    """
    if smi_results is None:
        print(f"Warning: No SMI results provided")
        return np.array([])
        
    if layer_name in smi_results and 'SMI' in smi_results[layer_name]:
        return smi_results[layer_name]['SMI']
    else:
        print(f"Warning: {layer_name} not found in data or no SMI values")
        return np.array([])



In [22]:
# =============================================================================
# MAIN ANALYSIS SCRIPT - AUTOMATED LOADING VERSION
# =============================================================================

# File paths for the SMI results
day1_filepath = r'D:\V1_SpatialModulation\2p\250517_JSY_JSY041_miniwalls_wolandmarks\TSeries-05172025-1438-001'
day3_filepath = r'D:\V1_SpatialModulation\2p\250517_JSY_JSY041_miniwalls_wolandmarks\TSeries-05172025-1438-001'
day5_filepath = r'D:\V1_SpatialModulation\2p\250517_JSY_JSY041_miniwalls_wolandmarks\TSeries-05172025-1438-001'

# Put all filepaths in a list
full_filepath = [day1_filepath, day3_filepath, day5_filepath]

# Initialize a list to store the results
recording_labels = [None] * len(full_filepath)
for i in range(np.size(recording_labels)):
    recording_labels[i] = f'Recording {i + 1}'
    
full_smi_results = [None] * len(full_filepath)
full_layer_smi_results = [None] * len(full_filepath)

print("=== LOADING SMI DATA ===\n")
for i in range(len(full_filepath)):
    # Check if the filepath exists
    if not os.path.exists(full_filepath[i]):
        print(f"Warning: Filepath does not exist: {full_filepath[i]}")
        full_smi_results[i] = None
    else:
        print(f"Loading {recording_labels[i]} from: {full_filepath[i]}")
        full_smi_results[i] = load_smi_file(full_filepath[i])
        
        if full_smi_results[i] is not None:
            print(f"  Successfully loaded {recording_labels[i]} data")
            # Initialize the dictionary for this recording
            full_layer_smi_results[i] = {}
            # Print available layers and cell counts
            for layer in ['L2/3', 'L4', 'L5', 'L6']:
                layer_data = extract_smi_data_from_results(full_smi_results[i], layer)
                full_layer_smi_results[i][layer] = layer_data
                if i == 0 and layer == 'L2/3':
                    print(layer_data)
                print(f"    {layer}: {len(layer_data)} cells")
        else:
            print(f"  Warning: No SMI data found for {recording_labels[i]}")
        print()

=== LOADING SMI DATA ===

Loading Recording 1 from: D:\V1_SpatialModulation\2p\250517_JSY_JSY041_miniwalls_wolandmarks\TSeries-05172025-1438-001
  Successfully loaded Recording 1 data
[ 0.41701834  0.38747421  0.41432579  0.72918227  0.78488241  0.63766733
  1.          1.          1.          0.28542764  0.71453101  0.04783177
  1.          0.37551373  0.56795899 -0.02762971  1.          0.52715959
  0.74644728  0.24766907  0.72748253  0.14064915  1.          0.40587458
  0.46180505  0.47140008  0.12229138  0.70452613  1.          0.81094971
  0.25041265  1.          0.56218641  0.73974204  1.          0.3654477
  0.28340334  1.          0.78511147  0.41267601  0.23778961  0.51167729
  0.16685997  0.11797637  1.          0.25775102  0.16571836  0.23905278
  0.76184466  0.56268487  0.50049575  0.4977422   0.40333177  0.89744403
  0.14329565  0.2254393   0.72855624  0.39831612  0.82822636  0.34943351
  0.49415647  0.27188182  0.45221001 -0.13666863  0.55435761  0.53323788
  0.03250367  

In [ ]:
full_layer_smi_mean = [None] * len(full_filepath)
full_layer_smi_median = [None] * len(full_filepath)
full_layer_smi_sem = [None] * len(full_filepath)
full_layer_smi_std = [None] * len(full_filepath)

for i in range(len(full_layer_smi_results)):
    full_layer_smi_mean[i] = {}
    full_layer_smi_median[i] = {}
    full_layer_smi_sem[i] = {}
    full_layer_smi_std[i] = {}
    
    for layer in ['L2/3', 'L4', 'L5', 'L6']:
        layer_data = full_layer_smi_results[i][layer]
        if len(layer_data) > 0:
            full_layer_smi_mean[i][layer] = np.mean(layer_data)
            full_layer_smi_median[i][layer] = np.median(layer_data)
            full_layer_smi_sem[i][layer] = stats.sem(layer_data)
            full_layer_smi_std[i][layer] = np.std(layer_data)
        else:
            full_layer_smi_mean[i][layer] = None
            full_layer_smi_median[i][layer] = None
            full_layer_smi_sem[i][layer] = None
            full_layer_smi_std[i][layer] = None
            

In [21]:
print(full_layer_smi_mean[0]["L6"])

0.4770433999543974


In [9]:
# =============================================================================
# REST OF YOUR ORIGINAL ANALYSIS CODE GOES HERE
# (calculate_layer_stats, perform_independent_comparison, plotting, etc.)
# =============================================================================

def calculate_layer_stats(data, layer_name, day):
    """Calculate descriptive statistics for each layer"""
    if len(data) == 0:
        return {
            'layer': layer_name,
            'day': day,
            'n': 0,
            'mean': np.nan,
            'sem': np.nan,
            'median': np.nan,
            'std': np.nan
        }
    
    return {
        'layer': layer_name,
        'day': day,
        'n': len(data),
        'mean': np.mean(data),
        'sem': stats.sem(data),
        'median': np.median(data),
        'std': np.std(data)
    }

def perform_independent_comparison(day1_data, day5_data, layer_name):
    """Perform independent samples t-test between days for a specific layer"""
    if len(day1_data) == 0 or len(day5_data) == 0:
        return {
            'layer': layer_name,
            't_stat': np.nan,
            'p_value': np.nan,
            'effect_size': np.nan,
            'mean_change': np.nan,
            'n_day1': len(day1_data),
            'n_day5': len(day5_data)
        }
    
    # Use Welch's t-test (unequal variances assumed)
    t_stat, p_value = ttest_ind(day5_data, day1_data, equal_var=False)
    
    # Calculate pooled standard deviation for effect size
    n1, n2 = len(day1_data), len(day5_data)
    if n1 > 1 and n2 > 1:
        pooled_std = np.sqrt(((n1-1)*np.var(day1_data, ddof=1) + (n2-1)*np.var(day5_data, ddof=1)) / (n1+n2-2))
        effect_size = (np.mean(day5_data) - np.mean(day1_data)) / pooled_std if pooled_std > 0 else np.nan
    else:
        effect_size = np.nan
    
    return {
        'layer': layer_name,
        't_stat': t_stat,
        'p_value': p_value,
        'effect_size': effect_size,
        'mean_change': np.mean(day5_data) - np.mean(day1_data),
        'n_day1': n1,
        'n_day5': n2
    }

def create_comparison_dataframe():
    """Create a long-format dataframe for analysis"""
    data = []
    
    # Day 1 data
    for val in day1_l23: data.append({'SMI': val, 'Layer': 'L2/3', 'Day': 1})
    for val in day1_l4: data.append({'SMI': val, 'Layer': 'L4', 'Day': 1})
    for val in day1_l5: data.append({'SMI': val, 'Layer': 'L5', 'Day': 1})
    for val in day1_l6: data.append({'SMI': val, 'Layer': 'L6', 'Day': 1})
    
    # Day 5 data
    for val in day5_l23: data.append({'SMI': val, 'Layer': 'L2/3', 'Day': 5})
    for val in day5_l4: data.append({'SMI': val, 'Layer': 'L4', 'Day': 5})
    for val in day5_l5: data.append({'SMI': val, 'Layer': 'L5', 'Day': 5})
    for val in day5_l6: data.append({'SMI': val, 'Layer': 'L6', 'Day': 5})
    
    return pd.DataFrame(data)

# Check if we have data to proceed with analysis
if len(day1_l23) == 0 and len(day1_l4) == 0 and len(day1_l5) == 0 and len(day1_l6) == 0:
    print("\nWarning: No Day 1 data found. Cannot proceed with analysis.")
elif len(day5_l23) == 0 and len(day5_l4) == 0 and len(day5_l5) == 0 and len(day5_l6) == 0:
    print(f"\nWarning: No {recording_labels[-1]} data found. Cannot proceed with analysis.")
else:
    print(f"\n=== PROCEEDING WITH ANALYSIS: {recording_labels[0]} vs {recording_labels[-1]} ===\n")
    
    # Continue with your original analysis code...
    # Calculate descriptive statistics
    print("=== DESCRIPTIVE STATISTICS ===\n")
    stats_day1 = [
        calculate_layer_stats(day1_l23, 'L2/3', 1),
        calculate_layer_stats(day1_l4, 'L4', 1),
        calculate_layer_stats(day1_l5, 'L5', 1),
        calculate_layer_stats(day1_l6, 'L6', 1)
    ]

    stats_day5 = [
        calculate_layer_stats(day5_l23, 'L2/3', 5),
        calculate_layer_stats(day5_l4, 'L4', 5),
        calculate_layer_stats(day5_l5, 'L5', 5),
        calculate_layer_stats(day5_l6, 'L6', 5)
    ]

    all_stats = stats_day1 + stats_day5
    stats_df = pd.DataFrame(all_stats)

    print("Layer Statistics by Day:")
    print(stats_df.round(3))

    # Perform independent samples comparisons between days
    print(f"\n=== INDEPENDENT SAMPLES COMPARISONS: {recording_labels[0]} vs {recording_labels[-1]} ===\n")
    independent_results = [
        perform_independent_comparison(day1_l23, day5_l23, 'L2/3'),
        perform_independent_comparison(day1_l4, day5_l4, 'L4'),
        perform_independent_comparison(day1_l5, day5_l5, 'L5'),
        perform_independent_comparison(day1_l6, day5_l6, 'L6')
    ]

    for result in independent_results:
        print(f"{result['layer']} ({recording_labels[0]}: n={result['n_day1']}, {recording_labels[-1]}: n={result['n_day5']}):")
        if not np.isnan(result['p_value']):
            print(f"  Mean difference ({recording_labels[-1]} - {recording_labels[0]}): {result['mean_change']:.3f}")
            print(f"  t = {result['t_stat']:.3f}, p = {result['p_value']:.4f}")
            print(f"  Effect size (Cohen's d): {result['effect_size']:.3f}")
            
            # Interpret significance and effect
            if result['p_value'] < 0.05:
                direction = "higher" if result['mean_change'] > 0 else "lower"
                print(f"  **SIGNIFICANT: {recording_labels[-1]} SMI {direction.upper()} than {recording_labels[0]}**")
            else:
                print("  No significant difference between days")
        else:
            print("  Cannot perform comparison (insufficient data)")
        print()

print("\n=== DATA LOADING AND EXTRACTION COMPLETE ===")
print("You can now add your additional analysis and visualization code...")


=== PROCEEDING WITH ANALYSIS: Day 1 vs Day 5 ===

=== DESCRIPTIVE STATISTICS ===

Layer Statistics by Day:
  layer  day    n   mean    sem  median    std
0  L2/3    1  110  0.483  0.028   0.459  0.295
1    L4    1   72  0.453  0.037   0.400  0.311
2    L5    1   51  0.445  0.045   0.392  0.315
3    L6    1   51  0.477  0.055   0.410  0.389
4  L2/3    5  110  0.483  0.028   0.459  0.295
5    L4    5   72  0.453  0.037   0.400  0.311
6    L5    5   51  0.445  0.045   0.392  0.315
7    L6    5   51  0.477  0.055   0.410  0.389

=== INDEPENDENT SAMPLES COMPARISONS: Day 1 vs Day 5 ===

L2/3 (Day 1: n=110, Day 5: n=110):
  Mean difference (Day 5 - Day 1): 0.000
  t = 0.000, p = 1.0000
  Effect size (Cohen's d): 0.000
  No significant difference between days

L4 (Day 1: n=72, Day 5: n=72):
  Mean difference (Day 5 - Day 1): 0.000
  t = 0.000, p = 1.0000
  Effect size (Cohen's d): 0.000
  No significant difference between days

L5 (Day 1: n=51, Day 5: n=51):
  Mean difference (Day 5 - Day 1):